In [1]:
# CELL 1: INSTALL DEPENDENCIES

%pip install -q wordcloud emoji
%pip install -q -U transformers accelerate
%pip install -q imbalanced-learn

In [2]:
# CELL 2: MOUNT DRIVE & PATHS

from google.colab import drive
drive.mount('/content/drive')

GLOVE_PATH = "/content/drive/MyDrive/glove.twitter.27B.100d.txt"
CSV_PATH   = "/content/drive/MyDrive/ulasan_aplikasi.csv"

In [3]:
# CELL 3: CONFIG & IMPORTS — OPTIMALISASI PENUH GPU T4

import re
import json
import time
import itertools
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter
import copy
import emoji
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn import CrossEntropyLoss
import torch.nn.functional as F
from transformers import EarlyStoppingCallback
import nltk
from nltk.tokenize import word_tokenize
from nltk.sentiment.vader import SentimentIntensityAnalyzer

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)
from sklearn.utils.class_weight import compute_class_weight

from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('vader_lexicon', quiet=True)

# Seed disamakan di numpy, torch, dan random agar eksperimen mudah direplikasi.
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

# Pilih GPU bila tersedia; semua model dan batch dipindahkan ke DEVICE ini.
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# OPTIMALISASI PENUH GPU T4 (16GB VRAM, Compute Capability 7.5)

# 1. TF32 matmul — T4 mendukung Tensor Float 32 untuk percepatan
# TF32 mempercepat operasi matrix multiplication di GPU NVIDIA tanpa overhead besar.
# Cocok untuk training deep learning karena sedikit penurunan presisi biasanya tidak signifikan.
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# 2. cuDNN benchmark — auto-tune kernel terbaik untuk input size tetap
# benchmark=True meminta cuDNN mencari kernel tercepat untuk bentuk input yang relatif stabil.
# deterministic=False dipilih sebagai trade-off: hasil bisa sedikit bervariasi, tetapi training lebih cepat.
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False  # non-deterministic = lebih cepat

# 3. Precision hint untuk matmul
# Hint presisi matmul menjaga performa tinggi saat PyTorch memilih implementasi operasi float32.
torch.set_float32_matmul_precision('high')

# 4. Optimalisasi jumlah thread CPU untuk data preprocessing
#    T4 Colab biasanya punya 2 vCPU
# Batasi thread CPU supaya preprocessing/DataLoader tidak berebut resource dengan runtime Colab.
# try/except dipakai karena jumlah thread tidak selalu bisa diubah setelah runtime aktif.
try:
    torch.set_num_interop_threads(2)
except RuntimeError:
    pass
try:
    torch.set_num_threads(2)
except RuntimeError:
    pass

# 5. Bersihkan cache GPU sebelum mulai
# Kosongkan cache allocator CUDA dari eksekusi sebelumnya agar baseline memori lebih bersih.
torch.cuda.empty_cache()

# 6. Tampilkan info GPU
# Info GPU dicetak di awal untuk memastikan notebook benar-benar memakai akselerator yang diharapkan.
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"   CUDA Cores: {torch.cuda.get_device_properties(0).multi_processor_count * 128}")
    print(f"   Compute Capability: {torch.cuda.get_device_properties(0).major}.{torch.cuda.get_device_properties(0).minor}")
    print(f"   PyTorch: {torch.__version__}")
else:
    print("GPU tidak tersedia, menggunakan CPU")

# Mapping label dibuat eksplisit agar urutan kelas konsisten di training, evaluasi, dan inference.
# ID2LABEL adalah kebalikannya untuk mengubah prediksi numerik kembali ke nama sentimen.
LABEL_MAP = {'negative': 0, 'neutral': 1, 'positive': 2}
ID2LABEL  = {v: k for k, v in LABEL_MAP.items()}

# Semua hasil eksperimen dikumpulkan di list ini agar tabel akhir dapat dibuat dari satu sumber.
experiment_results = []

# 7. Optimalisasi DataLoader
# pin_memory mempercepat transfer CPU→GPU; num_workers=2 sesuai batas umum Colab Free/T4.
DL_KWARGS = {'num_workers': 2, 'pin_memory': True} if torch.cuda.is_available() else {}

# Helper: banner & print terstandar
# Helper berikut menjaga format log konsisten sehingga progres antar model mudah dibandingkan.
def print_banner(title, device=DEVICE):
    """Fungsi untuk mencetak informasi device dan VRAM.
    
    Args:
        title (str): Judul yang akan ditampilkan
        device (torch.device): Device yang digunakan (default: DEVICE)
    """
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    vram     = f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "-"
    print(f"\n{title}")
    print(f"Device : {gpu_name} ({device})")
    print(f"VRAM   : {vram}")

# Dipakai untuk mengecek ukuran model yang benar-benar trainable, bukan semua parameter.
def count_parameters(model):
    """Menghitung jumlah parameter yang dapat dilatih dalam model.
    
    Args:
        model (nn.Module): Model PyTorch
        
    Returns:
        int: Jumlah total parameter yang dapat dilatih
    """
    return sum(p.numel() for p in model.parameters())

# Ringkasan per epoch dibuat padat agar perubahan loss, metrik, waktu, dan LR cepat terlihat.
def print_epoch(epoch, total, loss, metric_name, metric_val, elapsed, lr=None):
    """Mencetak informasi epoch training secara terformat.
    
    Args:
        epoch (int): Nomor epoch saat ini
        total (int): Total jumlah epoch
        loss (float): Nilai loss saat ini
        metric_name (str): Nama metrik yang digunakan
        metric_val (float): Nilai metrik saat ini
        elapsed (float): Waktu yang telah berlalu dalam detik
        lr (float, optional): Learning rate saat ini
    """
    lr_str = f" | LR: {lr:.6f}" if lr is not None else ""
    print(f"  Epoch [{epoch:>2}/{total}] | Loss: {loss:.4f} | "
          f"{metric_name}: {metric_val:.4f} | Time: {elapsed:.1f}s{lr_str}")

# Ringkasan best run memudahkan audit hyperparameter yang dipilih dari grid search.
def print_best(model_name, best_params, metric_name, metric_val, total_time):
    """Mencetak informasi model terbaik setelah training selesai.
    
    Args:
        model_name (str): Nama model
        best_params (dict): Parameter terbaik yang ditemukan
        metric_name (str): Nama metrik yang digunakan
        metric_val (float): Nilai metrik terbaik
        total_time (float): Total waktu training dalam detik
    """
    print(f"\n{model_name} — Best {metric_name}: {metric_val:.4f}")
    print(f"    Params : {best_params}")
    print(f"    Total  : {total_time:.1f}s\n")

In [4]:
# CELL 4: LOAD, CLEAN, LABEL + HYBRID SAMPLING

t0_load = time.time()

# Loader menerima dua kemungkinan nama kolom review agar notebook tetap kompatibel dengan dataset berbeda.
# NaN dan duplikat dibuang sejak awal supaya label otomatis tidak bias oleh data berulang.
def load_dataset(path):
    df = pd.read_csv(path)
    target_col = 'content' if 'content' in df.columns else 'Review'
    clean_df = df[[target_col]].dropna().drop_duplicates()
    clean_df.columns = ['content']
    return clean_df.reset_index(drop=True)

# Cleaning dibuat ringan: noise sosial/media dibuang, tetapi struktur kalimat utama tetap dipertahankan.
def clean_text_light(text):
    """Membersihkan teks ulasan dengan menghapus elemen yang tidak relevan.
    
    Args:
        text (str): Teks ulasan yang akan dibersihkan
        
    Returns:
        str: Teks yang sudah dibersihkan dari mentions, hashtags, retweets, URLs, dan emoji
    """
    text = re.sub(r'@[A-Za-z0-9_]+', '', text)
    text = re.sub(r'#[A-Za-z0-9_]+', '', text)
    text = re.sub(r'\bRT\b', '', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = emoji.replace_emoji(text, replace='')
    # Hapus karakter dekoratif/emotikon keyboard yang bisa mempengaruhi encoding
    text = re.sub(r'[═║╔╚╗╝╠╣╦╩⭐★✅✔⚠️📊📈⏹─━│┃▸■▪▫▶◆◇]+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# VADER dipakai sebagai weak labeler berbasis leksikon; cocok untuk baseline sentimen tanpa label manual.
sia = SentimentIntensityAnalyzer()

# Threshold kecil di sekitar nol memisahkan teks netral dari positif/negatif berdasarkan compound score.
def label_three_class(text, pos_thr=0.05, neg_thr=-0.05):
    """Melabeli teks menjadi tiga kelas sentimen berdasarkan skor VADER.
    
    Args:
        text (str): Teks yang akan dilabeli
        pos_thr (float, optional): Threshold untuk kelas positif. Default 0.05.
        neg_thr (float, optional): Threshold untuk kelas negatif. Default -0.05.
        
    Returns:
        tuple: (score, label) dimana score adalah skor sentimen dan label adalah
               'positive', 'neutral', atau 'negative'
    """
    score = sia.polarity_scores(text)['compound']
    if score >= pos_thr:   return score, 'positive'
    elif score <= neg_thr: return score, 'negative'
    return score, 'neutral'

# Pipeline label: load → clean untuk input model → skor VADER → mapping label numerik.
df_raw = load_dataset(CSV_PATH)
df_raw['text_clean'] = df_raw['content'].apply(clean_text_light)
scores_labels = df_raw['content'].apply(label_three_class)
df_raw['polarity_score'] = scores_labels.apply(lambda x: x[0])
df_raw['polarity']       = scores_labels.apply(lambda x: x[1])
df_raw['label']          = df_raw['polarity'].map(LABEL_MAP)

print(f"Total data: {df_raw.shape[0]}  |  Load: {time.time()-t0_load:.2f}s")
print("\nDistribusi AWAL:")
print(df_raw['polarity'].value_counts())

# STRATEGI DATA BARU: HYBRID SAMPLING
# Undersample mayoritas + Oversample minoritas ke target yang sama
# Target: 3000 per kelas (bukan 1195 seperti sebelumnya)
# Ini memberikan 3x lebih banyak data daripada sebelumnya

# Hybrid sampling menyeimbangkan kelas tanpa membuang semua kelas mayoritas atau menduplikasi minoritas mentah.
# Target per kelas dibuat sama agar loss dan metrik tidak didominasi label yang paling sering muncul.
def hybrid_sampling(df, label_col='label', target_per_class=3000, seed=42):
    """
    Hybrid sampling:
    - Kelas > target → undersample
    - Kelas < target → oversample (duplikasi + augmentasi ringan)
    - Kelas = target → tetap
    """
    # RNG lokal menjaga hasil sampling/augmentasi reproducible tanpa mengganggu seed global.
    rng = np.random.RandomState(seed)
    frames = []

    for label in sorted(df[label_col].unique()):
        subset = df[df[label_col] == label]
        n = len(subset)

        # Kelas mayoritas dipotong sampai target agar dataset seimbang dan training lebih cepat.
        if n >= target_per_class:
            # Undersample
            sampled = subset.sample(n=target_per_class, random_state=seed)
        # Kelas minoritas diperbanyak dengan sampling ulang karena data aslinya belum mencapai target.
        else:
            # Oversample: duplikasi + augmentasi ringan
            sampled = subset.copy()
            n_need = target_per_class - n
            extra_idx = rng.choice(subset.index, size=n_need, replace=True)
            extra = subset.loc[extra_idx].copy()

            # Augmentasi ringan: random word deletion (10% kata dihapus)
            # Duplikasi minoritas diberi variasi ringan supaya model tidak hanya menghafal teks yang sama.
            augmented_texts = []
            for text in extra['text_clean']:
                words = text.split()
                if len(words) > 3:
                    n_del = max(1, int(len(words) * 0.1))
                    for _ in range(n_del):
                        if len(words) > 2:
                            idx = rng.randint(0, len(words))
                            words.pop(idx)
                augmented_texts.append(' '.join(words))
            extra['text_clean'] = augmented_texts
            sampled = pd.concat([sampled, extra], ignore_index=True)

        frames.append(sampled)

    # Shuffle akhir mencampur semua kelas setelah sampling agar batch training tidak terurut per label.
    result = pd.concat(frames, ignore_index=True)
    result = result.sample(frac=1, random_state=seed).reset_index(drop=True)
    return result

# Target ini menjadi knob utama untuk menukar waktu training vs jumlah contoh per kelas.
TARGET_PER_CLASS = 3000  # 3x lebih banyak dari sebelumnya (1195)
df_raw = hybrid_sampling(df_raw, target_per_class=TARGET_PER_CLASS)

print(f"\nDistribusi SESUDAH hybrid sampling (target={TARGET_PER_CLASS}/kelas):")
print(df_raw['polarity'].value_counts())
print(f"Total: {df_raw.shape[0]}")

In [5]:
# CELL 5: HELPER UMUM + OPTIMALISASI DATALOADER UNTUK T4
# HELPER UMUM + TEXT AUGMENTATION + EARLY STOPPING

# Split bertingkat dibuat stratified supaya rasio kelas tetap konsisten di train/val/test.
def make_split(df, text_col, label_col, test_ratio, val_ratio_of_train=0.15, seed=RANDOM_STATE):
    """Membagi dataset menjadi train, validation, dan test set dengan stratifikasi.
    
    Args:
        df (pd.DataFrame): DataFrame yang berisi data
        text_col (str): Nama kolom yang berisi teks
        label_col (str): Nama kolom yang berisi label
        test_ratio (float): Proporsi data untuk test set
        val_ratio_of_train (float, optional): Proporsi data untuk validation set dari train set. Default 0.15.
        seed (int, optional): Random seed untuk reproduktibilitas. Default RANDOM_STATE.
        
    Returns:
    # Validation diambil dari trainfull untuk early stopping dan grid search.
        tuple: (X_train, X_val, X_test, y_train, y_val, y_test) - pembagian dataset
    """
    # Test dipisahkan lebih dulu agar tidak ikut memengaruhi pemilihan hyperparameter.
    X_trainfull, X_test, y_trainfull, y_test = train_test_split(
        df[text_col], df[label_col], test_size=test_ratio,
        stratify=df[label_col], random_state=seed
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_trainfull, y_trainfull, test_size=val_ratio_of_train,
        stratify=y_trainfull, random_state=seed
    )
    return X_train, X_val, X_test, y_train, y_val, y_test

# Class weight tetap dihitung meski sudah sampling seimbang sebagai perlindungan jika distribusi split berubah.
def get_class_weights(y_train_labels):
    """Menghitung bobot kelas untuk menangani imbalance class.
    
    Fungsi ini menggunakan metode 'balanced' dari sklearn untuk menghitung bobot kelas
    yang berbanding terbalik dengan frekuensi kelas. Bobot ini digunakan dalam loss function
    untuk memberikan penalti lebih besar pada kelas minoritas.
    
    Args:
        y_train_labels (array-like): Label dari training set
        
    Returns:
        torch.Tensor: Tensor bobot kelas yang dapat digunakan dalam loss function
    """
    # Bobot balanced memberi penalti lebih besar untuk kelas yang lebih sedikit di training split.
    weights = compute_class_weight('balanced',
                                   classes=np.array(list(LABEL_MAP.values())),
                                   y=y_train_labels)
    return torch.tensor(weights, dtype=torch.float)

# TEXT AUGMENTATION
# Augmentasi sengaja sederhana agar label sentimen tetap valid sambil menambah variasi urutan kata.
def augment_text(text, rng=None):
    """Melakukan augmentasi teks dengan random deletion dan random swap.
    
    Teknik augmentasi ini membantu meningkatkan variasi data training dengan:
    - Random deletion: menghapus 10% kata secara acak
    - Random swap: menukar posisi dua kata secara acak
    
    Args:
        text (str): Teks yang akan diaugmentasi
        rng (np.random.RandomState, optional): Random number generator. Default None.
        
    Returns:
        str: Teks yang sudah diaugmentasi
    """
    # RNG opsional memudahkan kontrol reproducibility saat augmentasi dipakai di Dataset.
    if rng is None:
        rng = np.random.RandomState()
    words = text.split()
    if len(words) <= 3:
        return text

    # Random deletion meniru variasi ulasan pendek/tidak lengkap tanpa mengubah mayoritas kata.
    # Random deletion (10%)
    if rng.random() < 0.5:
        n_del = max(1, int(len(words) * 0.1))
        for _ in range(n_del):
            if len(words) > 2:
                idx = rng.randint(0, len(words))
                words.pop(idx)

    # Random swap melatih model agar tidak terlalu sensitif pada posisi dua kata non-kritis.
    # Random swap (5%)
    if rng.random() < 0.5 and len(words) > 2:
        i, j = rng.choice(len(words), size=2, replace=False)
        words[i], words[j] = words[j], words[i]

    return ' '.join(words)

# EARLY STOPPING
# EarlyStopping menyimpan state terbaik sehingga model akhir bukan sekadar epoch terakhir.
class EarlyStopping:
    # patience menentukan toleransi epoch tanpa peningkatan; min_delta menghindari noise kecil dianggap membaik.
    def __init__(self, patience=3, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.best_state = None
        self.should_stop = False

    # Dipanggil setelah validasi; jika skor membaik, snapshot model disimpan di memori.
    def __call__(self, score, model):
        if self.best_score is None or score > self.best_score + self.min_delta:
            self.best_score = score
            self.best_state = copy.deepcopy(model.state_dict())
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True

    # Setelah loop berhenti, bobot terbaik dimuat kembali sebelum evaluasi test.
    def load_best(self, model):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)

# LABEL SMOOTHING LOSS
# Label smoothing mencegah model terlalu percaya diri pada weak label dari VADER.
class LabelSmoothingLoss(nn.Module):
    def __init__(self, num_classes=3, smoothing=0.1, weight=None):
        super().__init__()
        self.smoothing = smoothing
        self.num_classes = num_classes
        self.weight = weight

    # Target one-hot dilunakkan lalu dapat dikombinasikan dengan class weight per contoh.
    def forward(self, pred, target):
        log_prob = F.log_softmax(pred, dim=-1)
        one_hot = torch.zeros_like(pred).scatter(1, target.unsqueeze(1), 1)
        one_hot = one_hot * (1 - self.smoothing) + self.smoothing / self.num_classes
        loss = -(one_hot * log_prob).sum(dim=-1)
        if self.weight is not None:
            w = self.weight[target]
            loss = loss * w
        return loss.mean()

def log_result(model_name, split_info, feature_info,
               y_train_true, y_train_pred, y_test_true, y_test_pred, best_params):
    result = {
        "Model": model_name, "Split": split_info, "Fitur": feature_info,
        "Best Params": str(best_params),
        "Train Accuracy": accuracy_score(y_train_true, y_train_pred),
        "Test Accuracy": accuracy_score(y_test_true, y_test_pred),
        "Precision": precision_score(y_test_true, y_test_pred, average='macro', zero_division=0),
        "Recall": recall_score(y_test_true, y_test_pred, average='macro', zero_division=0),
        "F1-Score": f1_score(y_test_true, y_test_pred, average='macro', zero_division=0),
    }
    experiment_results.append(result)
    return result

def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=list(LABEL_MAP.keys()),
                yticklabels=list(LABEL_MAP.keys()))
    plt.title(title); plt.xlabel('Prediksi'); plt.ylabel('Aktual')
    plt.tight_layout(); plt.show()

def build_vocab(texts, min_freq=2):
    counter = Counter()
    for t in texts:
        counter.update(word_tokenize(t.lower()))
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)
    return vocab

def text_to_ids(text, vocab, max_len):
    tokens = word_tokenize(text.lower())[:max_len]
    ids = [vocab.get(t, vocab['<UNK>']) for t in tokens]
    ids += [vocab['<PAD>']] * (max_len - len(ids))
    return ids

In [6]:
# CELL 6: MODEL A — TextCNN, Split 80:20
# (DITINGKATKAN: Early Stop + LR Scheduler + Label Smoothing + Augmentasi + Dropout lebih tinggi)

torch.cuda.empty_cache()
# Split 80:20 memberi data train lebih banyak untuk model CNN yang belajar embedding dari nol.
SPLIT_TEST_RATIO_A = 0.20

print_banner("MODEL A — TextCNN  (Split 80:20)")

X_tr_A, X_val_A, X_te_A, y_tr_A, y_val_A, y_te_A = make_split(
    df_raw, 'text_clean', 'label', test_ratio=SPLIT_TEST_RATIO_A,
    val_ratio_of_train=0.15  # validasi lebih besar
)

# MAX_LEN membatasi jumlah token agar batch padat dan konvolusi tetap efisien di GPU.
MAX_LEN_A = 50  # sedikit lebih panjang
vocab_A = build_vocab(X_tr_A, min_freq=2)
VOCAB_SIZE_A = len(vocab_A)
print(f"  Vocab: {VOCAB_SIZE_A} | Train: {len(X_tr_A)} | Val: {len(X_val_A)} | Test: {len(X_te_A)}")

# Dataset TextCNN mengubah teks menjadi indeks vocabulary dan menerapkan augmentasi hanya saat training.
# TextCNN memakai beberapa ukuran kernel untuk menangkap pola frasa pendek hingga agak panjang.
class TextCNNDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len, augment=False):
        self.texts  = list(texts)
        self.vocab  = vocab
        self.max_len = max_len
        self.labels = labels.values if hasattr(labels, 'values') else labels
        self.augment = augment
        self.rng = np.random.RandomState(RANDOM_STATE)
        # Pre-tokenize jika tidak augment
        # Tanpa augmentasi, tokenisasi disimpan sekali untuk mengurangi overhead saat validasi/test.
        if not augment:
            self.ids = [text_to_ids(t, vocab, max_len) for t in texts]
        else:
            self.ids = None

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        # Augmentasi probabilistik menjaga sebagian besar batch tetap berisi teks asli.
        if self.augment and self.rng.random() < 0.3:  # 30% chance augment
            text = augment_text(self.texts[idx], self.rng)
        else:
            text = self.texts[idx]
        ids = text_to_ids(text, self.vocab, self.max_len)
        return (torch.tensor(ids, dtype=torch.long),
                torch.tensor(self.labels[idx], dtype=torch.long))

class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, num_filters=100,
                 kernel_sizes=(2, 3, 4, 5), num_classes=3, dropout=0.4):
        super().__init__()
        # Embedding trainable belajar representasi khusus domain ulasan game.
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        # Conv1d paralel bekerja seperti n-gram detector dengan ukuran jendela berbeda.
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, kernel_size=k) for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), num_classes)

    # Forward: embedding → convolution → max-pooling global → classifier.
    def forward(self, x):
        emb = self.dropout(self.embedding(x)).permute(0, 2, 1)  # dropout di embedding
        # ReLU mempertahankan fitur frasa yang aktif kuat untuk tiap filter.
        conv_outs = [torch.relu(conv(emb)) for conv in self.convs]
        # Max-pooling mengambil sinyal terkuat dari tiap filter tanpa bergantung posisi token.
        pooled = [torch.max(c, dim=2)[0] for c in conv_outs]
        concat = self.dropout(torch.cat(pooled, dim=1))
        return self.fc(concat)

# Fungsi training mengisolasi satu kombinasi hyperparameter agar grid search mudah dijalankan.
def train_textcnn(num_filters, dropout, lr, epochs=15, patience=4):
    """Melatih model TextCNN dengan hyperparameter tertentu.
    
    Fungsi ini melakukan training loop untuk model TextCNN dengan fitur:
    - Label smoothing loss untuk regularisasi
    - Class weights untuk menangani imbalance
    - Cosine annealing learning rate scheduler
    - Gradient scaling untuk mixed precision training
    - Early stopping untuk mencegah overfitting
    
    Args:
        num_filters (int): Jumlah filter per convolution
        dropout (float): Tingkat dropout
        lr (float): Learning rate
        epochs (int, optional): Jumlah maksimum epoch. Default 15.
        patience (int, optional): Patience untuk early stopping. Default 4.
        
    Returns:
        tuple: (model, best_val_acc) - model terbaik dan akurasi validasi terbaik
    """
    model = TextCNN(VOCAB_SIZE_A, num_filters=num_filters,
                    dropout=dropout, kernel_sizes=(2,3,4,5)).to(DEVICE)
    # Class weights dihitung dari train split agar loss mengikuti distribusi data yang benar-benar dilatih.
    class_weights = get_class_weights(y_tr_A.values).to(DEVICE)

    # Label Smoothing + Weighted
    criterion = LabelSmoothingLoss(num_classes=3, smoothing=0.1, weight=class_weights)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)  # weight decay lebih besar

    # Cosine Annealing LR Scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=3, T_mult=2, eta_min=1e-5
    )

    # GradScaler menstabilkan mixed precision sehingga training lebih cepat tanpa underflow gradien.
    scaler = torch.amp.GradScaler('cuda')
    early_stop = EarlyStopping(patience=patience, min_delta=0.002)

    # Batch train lebih kecil karena ada backward pass; val lebih besar karena hanya inference.
    train_loader = DataLoader(
        TextCNNDataset(X_tr_A, y_tr_A, vocab_A, MAX_LEN_A, augment=True),
        batch_size=64, shuffle=True, **DL_KWARGS
    )
    val_loader = DataLoader(
        TextCNNDataset(X_val_A, y_val_A, vocab_A, MAX_LEN_A, augment=False),
        batch_size=128, **DL_KWARGS
    )

    best_val_acc = 0
    # Loop epoch berhenti lebih awal jika validasi tidak lagi membaik.
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        t_ep = time.time()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'):
                loss = criterion(model(xb), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            # Gradient clipping membatasi update ekstrem yang sering muncul saat mixed precision.
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * xb.size(0)

        # Scheduler cosine menurunkan/menaikkan LR secara periodik agar optimizer tidak mudah stagnan.
        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']

        # Validasi
        model.eval()
        val_preds, val_true = [], []
        with torch.no_grad(), torch.amp.autocast('cuda'):
            for xb, yb in val_loader:
                preds = torch.argmax(model(xb.to(DEVICE, non_blocking=True)), dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_true.extend(yb.numpy())

        avg_loss = running_loss / len(train_loader.dataset)
        val_acc = accuracy_score(val_true, val_preds)
        print_epoch(epoch, epochs, avg_loss, "Val Acc", val_acc,
                    time.time()-t_ep, lr=current_lr)

        # Early Stopping
        early_stop(val_acc, model)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
        if early_stop.should_stop:
            print(f"  Early stopping di epoch {epoch}")
            break

    early_stop.load_best(model)
    return model, best_val_acc

# --- Grid Search (lebih fokus) ---
# Grid dibuat kecil dan fokus agar eksplorasi selesai realistis di Colab T4.
param_grid_A = {
    'num_filters': [100, 150],
    'dropout': [0.3, 0.4],
    'lr': [5e-4, 1e-3]
}
best_val_acc_A, best_model_A, best_params_A = -1, None, None
t_start_A = time.time()

# Setiap kombinasi dilatih independen; metrik validasi menentukan model yang dipakai untuk test.
for nf, do, lr in itertools.product(
    param_grid_A['num_filters'], param_grid_A['dropout'], param_grid_A['lr']
):
    print(f"\n num_filters={nf}, dropout={do}, lr={lr}")
    model, val_acc = train_textcnn(nf, do, lr, epochs=15, patience=4)
    if val_acc > best_val_acc_A:
        best_val_acc_A, best_model_A = val_acc, model
        best_params_A = {'num_filters': nf, 'dropout': do, 'lr': lr}

print_best("Model A: TextCNN", best_params_A, "Val Acc", best_val_acc_A,
           time.time()-t_start_A)

# Helper prediksi dipakai ulang oleh TextCNN, BiLSTM, dan ensemble agar evaluasi konsisten.
def predict_dl_model(model, dataset_cls, texts, vocab=None, max_len=None, w2v=None):
    dummy_labels = pd.Series(np.zeros(len(texts)))
    if vocab is not None:
        loader = DataLoader(dataset_cls(texts, dummy_labels, vocab, max_len, augment=False),
                            batch_size=128, **DL_KWARGS)
    else:
        loader = DataLoader(dataset_cls(texts, dummy_labels, w2v),
                            batch_size=128, **DL_KWARGS)
    model.eval()
    preds = []
    with torch.no_grad(), torch.amp.autocast('cuda'):
        for xb, _ in loader:
            preds.extend(torch.argmax(model(xb.to(DEVICE, non_blocking=True)), dim=1).cpu().numpy())
    return preds

y_pred_train_A = predict_dl_model(best_model_A, TextCNNDataset, X_tr_A, vocab=vocab_A, max_len=MAX_LEN_A)
y_pred_test_A  = predict_dl_model(best_model_A, TextCNNDataset, X_te_A, vocab=vocab_A, max_len=MAX_LEN_A)

res_A = log_result("Model A: TextCNN",
                    f"{int((1-SPLIT_TEST_RATIO_A)*100)}:{int(SPLIT_TEST_RATIO_A*100)}",
                    "Trainable Embedding + Augmentasi + Label Smoothing",
                    y_tr_A, y_pred_train_A, y_te_A, y_pred_test_A, best_params_A)
print(f"  Test Acc: {res_A['Test Accuracy']:.4f} | F1: {res_A['F1-Score']:.4f}")
plot_confusion(y_te_A, y_pred_test_A, "Confusion Matrix — Model A (TextCNN)")

In [7]:
# CELL 7: MODEL B — BiLSTM + GloVe + ATTENTION, Split 70:30

torch.cuda.empty_cache()
# Split 70:30 menguji BiLSTM pada test set lebih besar untuk melihat generalisasi urutan.
SPLIT_TEST_RATIO_B = 0.30

print_banner("MODEL B — BiLSTM + GloVe + Attention (Split 70:30)")

X_tr_B, X_val_B, X_te_B, y_tr_B, y_val_B, y_te_B = make_split(
    df_raw, 'text_clean', 'label', test_ratio=SPLIT_TEST_RATIO_B,
    val_ratio_of_train=0.15
)

# Panjang token disamakan dengan TextCNN agar perbandingan model non-transformer tetap adil.
MAX_LEN_B = 50
EMBED_DIM_B = 100
vocab_B = build_vocab(X_tr_B, min_freq=2)
VOCAB_SIZE_B = len(vocab_B)
print(f"  Vocab: {VOCAB_SIZE_B} | Train: {len(X_tr_B)} | Val: {len(X_val_B)} | Test: {len(X_te_B)}")

# GloVe memberi embedding awal berbasis korpus besar sehingga BiLSTM tidak belajar representasi dari nol.
def load_glove_embeddings(glove_path, vocab, embed_dim):
    # Kata yang tidak ada di GloVe diberi nilai acak kecil; PAD dibuat nol agar tidak membawa sinyal.
    embedding_matrix = np.random.uniform(-0.05, 0.05, (len(vocab), embed_dim)).astype(np.float32)
    embedding_matrix[vocab['<PAD>']] = np.zeros(embed_dim)
    found = 0
    # Hanya vektor untuk kata di vocabulary yang dimuat agar memori tetap hemat.
    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip().split(' ')
            word = parts[0]
            if word in vocab:
                embedding_matrix[vocab[word]] = np.array(parts[1:], dtype=np.float32)
                found += 1
    print(f"  GloVe coverage: {found}/{len(vocab)} ({found/len(vocab)*100:.1f}%)")
    return torch.tensor(embedding_matrix, dtype=torch.float32)

glove_embedding_matrix = load_glove_embeddings(GLOVE_PATH, vocab_B, EMBED_DIM_B)

# Dataset BiLSTM memakai vocabulary yang sama konsepnya dengan TextCNN dan augmentasi hanya untuk train.
class BiLSTMDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=MAX_LEN_B, augment=False):
        self.texts = list(texts)
        self.vocab = vocab
        self.max_len = max_len
        self.labels = labels.values if hasattr(labels, 'values') else labels
        self.augment = augment
        self.rng = np.random.RandomState(RANDOM_STATE)
        # Cache ids untuk mode non-augment mengurangi biaya tokenisasi berulang saat evaluasi.
        if not augment:
            self.ids = [text_to_ids(t, vocab, max_len) for t in texts]
        else:
            self.ids = None

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        # Probabilitas 30% memberi regularisasi tanpa membuat distribusi teks terlalu berbeda.
        if self.augment and self.rng.random() < 0.3:
            text = augment_text(self.texts[idx], self.rng)
        else:
            text = self.texts[idx]
        ids = text_to_ids(text, self.vocab, self.max_len)
        return (torch.tensor(ids, dtype=torch.long),
                torch.tensor(self.labels[idx], dtype=torch.long))

# BiLSTM + Self-Attention
# BiLSTM membaca konteks kiri-kanan; attention memilih token/fragmen yang paling relevan untuk sentimen.
class AttentionBiLSTM(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim=128, num_layers=1,
                 num_classes=3, dropout=0.4, freeze_embed=False):
        super().__init__()
        vocab_size, embed_dim = embedding_matrix.shape
        # Embedding dapat di-freeze atau fine-tune untuk menukar stabilitas vs adaptasi domain.
        self.embedding = nn.Embedding.from_pretrained(
            embedding_matrix, freeze=freeze_embed, padding_idx=0)
        self.embed_drop = nn.Dropout(0.2)
        # Bidirectional LSTM menangkap dependensi sebelum dan sesudah kata target.
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, bidirectional=True,
                            dropout=dropout if num_layers > 1 else 0.0)

        # Self-Attention
        # Attention menghasilkan bobot per timestep untuk merangkum sequence menjadi satu context vector.
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1, bias=False)
        )

        # Head MLP dengan dropout mengubah context vector menjadi logits tiga kelas.
        self.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden_dim, num_classes)
        )

    # Forward: embedding GloVe → BiLSTM → attention pooling → classifier.
    def forward(self, x):
        emb = self.embed_drop(self.embedding(x))
        lstm_out, _ = self.lstm(emb)  # (batch, seq, hidden*2)

        # Attention weights
        attn_weights = self.attention(lstm_out).squeeze(-1)  # (batch, seq)
        attn_weights = F.softmax(attn_weights, dim=-1)
        # Weighted sum memberi representasi kalimat yang fokus pada token berbobot tinggi.
        context = torch.bmm(attn_weights.unsqueeze(1), lstm_out).squeeze(1)

        return self.fc(context)

# Fungsi training BiLSTM dibuat paralel dengan TextCNN agar strategi regularisasi konsisten.
def train_bilstm(hidden_dim, num_layers, lr, freeze_embed, epochs=12, patience=4):
    """Melatih model BiLSTM dengan self-attention menggunakan hyperparameter tertentu.
    
    Fungsi ini melakukan training loop untuk model BiLSTM dengan fitur:
    - Label smoothing loss untuk regularisasi
    - Class weights untuk menangani imbalance
    - Cosine annealing learning rate scheduler
    - Gradient scaling untuk mixed precision training
    - Early stopping untuk mencegah overfitting
    
    Args:
        hidden_dim (int): Dimensi hidden state LSTM
        num_layers (int): Jumlah layer LSTM
        lr (float): Learning rate
        freeze_embed (bool): Apakah freeze embedding layer
        epochs (int, optional): Jumlah maksimum epoch. Default 12.
        patience (int, optional): Patience untuk early stopping. Default 4.
        
    Returns:
        tuple: (model, best_val_acc) - model terbaik dan akurasi validasi terbaik
    """
    model = AttentionBiLSTM(glove_embedding_matrix, hidden_dim=hidden_dim,
                            num_layers=num_layers, freeze_embed=freeze_embed,
                            dropout=0.4).to(DEVICE)
    class_weights = get_class_weights(y_tr_B.values).to(DEVICE)
    criterion = LabelSmoothingLoss(num_classes=3, smoothing=0.1, weight=class_weights)

    # AdamW + weight decay membantu regularisasi parameter LSTM dan classification head.
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=3, T_mult=2, eta_min=1e-5
    )
    scaler = torch.amp.GradScaler('cuda')
    early_stop = EarlyStopping(patience=patience, min_delta=0.002)

    # Train memakai augmentasi; validation memakai teks asli agar metrik mencerminkan data nyata.
    train_loader = DataLoader(
        BiLSTMDataset(X_tr_B, y_tr_B, vocab_B, MAX_LEN_B, augment=True),
        batch_size=64, shuffle=True, **DL_KWARGS
    )
    val_loader = DataLoader(
        BiLSTMDataset(X_val_B, y_val_B, vocab_B, MAX_LEN_B, augment=False),
        batch_size=128, **DL_KWARGS
    )

    best_val_acc = 0
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        t_ep = time.time()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'):
                loss = criterion(model(xb), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            # Clipping penting untuk RNN karena gradien dapat melonjak pada sequence tertentu.
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * xb.size(0)

        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']

        model.eval()
        val_preds, val_true = [], []
        with torch.no_grad(), torch.amp.autocast('cuda'):
            for xb, yb in val_loader:
                preds = torch.argmax(model(xb.to(DEVICE, non_blocking=True)), dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_true.extend(yb.numpy())

        avg_loss = running_loss / len(train_loader.dataset)
        val_acc = accuracy_score(val_true, val_preds)
        print_epoch(epoch, epochs, avg_loss, "Val Acc", val_acc,
                    time.time()-t_ep, lr=current_lr)

        early_stop(val_acc, model)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
        if early_stop.should_stop:
            print(f"  Early stopping di epoch {epoch}")
            break

    early_stop.load_best(model)
    return model, best_val_acc

# --- Grid Search ---
# Grid BiLSTM difokuskan pada jumlah layer dan LR karena embedding sudah dibantu GloVe.
param_grid_B = {
    'hidden_dim': [128],
    'num_layers': [1, 2],
    'lr': [5e-4, 1e-3],
    'freeze_embed': [False]
}
best_val_acc_B, best_model_B, best_params_B = -1, None, None
t_start_B = time.time()

# Kombinasi terbaik dipilih dari validation accuracy sebelum dievaluasi di test set.
for hd, nl, lr, fe in itertools.product(
    param_grid_B['hidden_dim'], param_grid_B['num_layers'],
    param_grid_B['lr'], param_grid_B['freeze_embed']
):
    print(f"\n hidden={hd}, layers={nl}, lr={lr}, freeze={fe}")
    model, val_acc = train_bilstm(hd, nl, lr, fe, epochs=12, patience=4)
    if val_acc > best_val_acc_B:
        best_val_acc_B, best_model_B = val_acc, model
        best_params_B = {'hidden_dim': hd, 'num_layers': nl, 'lr': lr, 'freeze_embed': fe}

print_best("Model B: BiLSTM+Attn+GloVe", best_params_B, "Val Acc", best_val_acc_B,
           time.time()-t_start_B)

y_pred_train_B = predict_dl_model(best_model_B, BiLSTMDataset, X_tr_B, vocab=vocab_B, max_len=MAX_LEN_B)
y_pred_test_B  = predict_dl_model(best_model_B, BiLSTMDataset, X_te_B, vocab=vocab_B, max_len=MAX_LEN_B)

res_B = log_result("Model B: BiLSTM+Attn+GloVe",
                    f"{int((1-SPLIT_TEST_RATIO_B)*100)}:{int(SPLIT_TEST_RATIO_B*100)}",
                    "GloVe + Self-Attention + Label Smoothing",
                    y_tr_B, y_pred_train_B, y_te_B, y_pred_test_B, best_params_B)
print(f"  Test Acc: {res_B['Test Accuracy']:.4f} | F1: {res_B['F1-Score']:.4f}")
plot_confusion(y_te_B, y_pred_test_B, "Confusion Matrix — Model B (BiLSTM+Attn)")

In [8]:
# CELL 8: Model C — RoBERTa Fine-tuning
# Target: Colab Free GPU T4 (16 GB VRAM, ~10 TFLOPS FP16)

import math, gc, copy

# Bersihkan cache dan garbage collector karena fine-tuning transformer sangat sensitif terhadap VRAM.
torch.cuda.empty_cache()
gc.collect()

# Split 75:25 menjadi acuan utama untuk RoBERTa dan ensemble akhir.
SPLIT_TEST_RATIO_C = 0.25
print_banner("MODEL C — RoBERTa Fine-tuned (Split 75:25)")

X_tr_C, X_val_C, X_te_C, y_tr_C, y_val_C, y_te_C = make_split(
    df_raw, 'text_clean', 'label',
    test_ratio=SPLIT_TEST_RATIO_C,
    val_ratio_of_train=0.15
)

# Model CardiffNLP dipilih karena sudah dilatih pada data Twitter/sosial yang mirip gaya ulasan singkat.
MODEL_NAME_C = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer_C  = AutoTokenizer.from_pretrained(MODEL_NAME_C)

# Tokenisasi batch besar + ukur panjang aktual
# Tokenisasi dibuat sebagai fungsi agar train/val/test memakai padding dan truncation yang sama.
def tokenize_texts(texts, max_length=128):
    # padding=max_length membuat tensor berukuran tetap sehingga batching di Trainer lebih sederhana.
    return tokenizer_C(
        list(texts),
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )

# MAX_LEN ditentukan dari p95 panjang token agar mayoritas teks utuh tanpa memboroskan VRAM.
sample_lengths = [len(tokenizer_C.encode(t)) for t in X_tr_C[:500]]
p95_len = int(np.percentile(sample_lengths, 95))
MAX_LEN_C = min(128, max(64, p95_len + 8))
print(f"  p95 token length = {p95_len} → max_length = {MAX_LEN_C}")

t_tok = time.time()
# Encoding dilakukan sekali di awal agar Trainer tidak mengulang tokenisasi pada setiap epoch.
train_enc_C = tokenize_texts(X_tr_C, MAX_LEN_C)
val_enc_C   = tokenize_texts(X_val_C, MAX_LEN_C)
test_enc_C  = tokenize_texts(X_te_C, MAX_LEN_C)
print(f"  Tokenization: {time.time()-t_tok:.1f}s")
print(f"  Train: {len(X_tr_C)} | Val: {len(X_val_C)} | Test: {len(X_te_C)}")

# Dataset (tidak berubah, sudah efisien)
# Dataset HuggingFace mengembalikan dict sesuai format Trainer: input_ids, attention_mask, labels.
class HFDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = torch.tensor(
            labels.values if hasattr(labels, 'values') else labels,
            dtype=torch.long
        )
    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels':         self.labels[idx],
        }
    def __len__(self):
        return len(self.labels)

train_dataset_C = HFDataset(train_enc_C, y_tr_C)
val_dataset_C   = HFDataset(val_enc_C,   y_val_C)
test_dataset_C  = HFDataset(test_enc_C,  y_te_C)

# Class weights dipakai di custom Trainer untuk menjaga loss tetap sensitif pada tiap kelas.
class_weights_C = get_class_weights(y_tr_C.values)

# Mean-Pooling Head: lebih stabil dari [CLS] saja
# Mean pooling memanfaatkan semua token non-padding, bukan hanya token awal, sehingga lebih stabil untuk ulasan.
class RoBERTaMeanPoolHead(nn.Module):
    """Arsitektur RoBERTa dengan mean-pooling head untuk klasifikasi.
    
    Model ini menggunakan pretrained RoBERTa encoder dengan custom classification head:
    - Mean-pooling dari hidden states untuk mendapatkan representasi kalimat
    - Dropout untuk regularisasi
    - Linear layer untuk klasifikasi
    - Inisialisasi Xavier untuk stabilitas training
    
    Args:
        base_model (AutoModel): Pretrained RoBERTa model
        num_labels (int, optional): Jumlah kelas. Default 3.
        dropout_rate (float, optional): Tingkat dropout. Default 0.2.
    """
    def __init__(self, base_model, num_labels=3, dropout_rate=0.2):
        super().__init__()
        # Hanya encoder RoBERTa dipakai; classification head bawaan diganti dengan head sederhana.
        self.roberta    = base_model.roberta          # encoder saja
        self.dropout    = nn.Dropout(dropout_rate)     # dropout eksplisit
        self.classifier = nn.Linear(768, num_labels)
        # inisialisasi head agar tidak mulai dari random besar
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kw):
        """Forward pass model RoBERTaMeanPoolHead.
        
        Args:
            input_ids (torch.Tensor): Input tensor dengan shape (batch_size, seq_len)
            attention_mask (torch.Tensor): Attention mask dengan shape (batch_size, seq_len)
            labels (torch.Tensor, optional): Label ground truth
            **kw: Argumen tambahan
            
        Returns:
            dict: Dictionary berisi 'loss' dan 'logits' jika labels diberikan,
                  atau hanya 'logits' jika tidak ada labels
        """
        outputs = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        # attention_mask memastikan token padding tidak ikut dihitung dalam rata-rata representasi.
        hidden = outputs.last_hidden_state                       # (B, L, 768)
        mask_exp = attention_mask.unsqueeze(-1).float()          # (B, L, 1)
        pooled = (hidden * mask_exp).sum(1) / mask_exp.sum(1).clamp(min=1e-9)
        pooled = self.dropout(pooled)                            # (B, 768)
        logits = self.classifier(pooled)                         # (B, num_labels)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels)               # placeholder
        return {"loss": loss, "logits": logits} if loss is not None else {"logits": logits}


# Layer-wise Learning Rate Decay (LLRD)
# Layer bawah sudah matang → LR kecil; head → LR besar
# LLRD memberi LR berbeda per layer: layer bawah dijaga stabil, layer atas/head lebih cepat beradaptasi.
def build_llrd_optimizer(model, base_lr=2e-5, weight_decay=0.03, lr_decay=0.9):
    # Bias dan LayerNorm tidak diberi weight decay agar statistik normalisasi tidak terdistorsi.
    no_decay_keys = ["bias", "LayerNorm.weight", "LayerNorm.bias"]
    param_groups  = []

    # Setiap parameter ditempatkan di param group sendiri supaya LR multiplier bisa presisi.
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        # tentukan multiplier per komponen
        if "classifier" in name:
            lr_mult = 10.0                          # head → paling besar
        elif "pooler" in name:
            lr_mult = 5.0
        elif "encoder.layer." in name:
            layer_num = int(name.split("encoder.layer.")[1].split(".")[0])
            lr_mult = lr_decay ** (11 - layer_num)  # layer 11→1.0, layer 0→≈0.28
        else:
            lr_mult = lr_decay ** 12                # embeddings → paling kecil

        param_groups.append({
            "params":       [param],
            "lr":           base_lr * lr_mult,
            "weight_decay": 0.0 if any(nd in name for nd in no_decay_keys) else weight_decay,
        })

    return torch.optim.AdamW(param_groups)


# Trainer dengan Label Smoothing + Class Weights + LLRD
# Custom Trainer menggabungkan label smoothing, class weight, dan optimizer LLRD dalam API HuggingFace.
class WeightedLabelSmoothingTrainer(Trainer):
    def __init__(self, *args, class_weights=None, smoothing=0.05,
                 base_lr=2e-5, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.smoothing     = smoothing
        self.base_lr       = base_lr                     # untuk LLRD

    # Override optimizer → pakai LLRD
    # Override ini memastikan Trainer memakai optimizer LLRD, bukan AdamW default.
    def create_optimizer(self):
        if self.optimizer is None:
            self.optimizer = build_llrd_optimizer(
                self.model,
                base_lr=self.base_lr,
                weight_decay=self.args.weight_decay,
            )
        return self.optimizer

    # Loss custom mengganti cross-entropy standar agar weak label tidak terlalu dipelajari secara kaku.
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits if hasattr(outputs, 'logits') else outputs["logits"]
        num_labels = logits.size(-1)

        log_prob = F.log_softmax(logits, dim=-1)
        one_hot  = torch.zeros_like(logits).scatter(1, labels.unsqueeze(1), 1)
        one_hot  = one_hot * (1 - self.smoothing) + self.smoothing / num_labels

        # Weight per sample diambil dari label asli sehingga kelas minoritas mendapat penalti lebih besar.
        if self.class_weights is not None:
            w    = self.class_weights.to(logits.device)[labels]
            loss = -(one_hot * log_prob).sum(dim=-1) * w
        else:
            loss = -(one_hot * log_prob).sum(dim=-1)
        loss = loss.mean()

        return (loss, outputs) if return_outputs else loss


# Macro F1 dipantau karena lebih informatif daripada accuracy saat performa antar kelas tidak seimbang.
def compute_metrics(eval_pred):
    """Menghitung metrik evaluasi untuk model.
    
    Args:
        eval_pred (tuple): Tuple berisi (logits, labels)
        
    Returns:
        dict: Dictionary berisi metrik 'accuracy' dan 'f1_macro'
    """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average='macro', zero_division=0),
    }

# 3 combo dengan variasi freeze & LR
# Kombinasi hyperparameter untuk fine-tuning RoBERTa dengan pendekatan berbeda:
# 1. Freeze 4 layer bawah, LR 2e-5, batch size 16
# 2. Freeze 6 layer bawah, LR 3e-5, batch size 16
# 3. Freeze 4 layer bawah, LR 1.5e-5, batch size 32 (batch size lebih besar)
# Combo RoBERTa sengaja sedikit karena setiap fine-tuning mahal; variasinya fokus pada LR, batch, warmup, freeze.
param_combos_C = [
    {'learning_rate': 2e-5,  'batch_size': 16, 'warmup_ratio': 0.10,
     'epochs': 5, 'freeze_layers': 4},                   # 5 epoch, freeze 4
    {'learning_rate': 3e-5,  'batch_size': 16, 'warmup_ratio': 0.06,
     'epochs': 4, 'freeze_layers': 6},                   # alternatif
    {'learning_rate': 1.5e-5,'batch_size': 32, 'warmup_ratio': 0.10,
     'epochs': 5, 'freeze_layers': 4},                   # bs besar, LR kecil
]

best_val_f1_C, best_trainer_C, best_params_C = -1, None, None
t_start_C = time.time()

# Load base model SEKALI
# Base model dimuat sekali lalu di-deepcopy per combo untuk menghemat waktu download/loading.
base_model_C = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_C, num_labels=3, ignore_mismatched_sizes=True
)

# Tiap combo dimulai dari bobot pretrained yang sama agar perbandingan hyperparameter adil.
for combo in param_combos_C:
    lr   = combo['learning_rate']
    bs   = combo['batch_size']
    wr   = combo['warmup_ratio']
    ep   = combo['epochs']
    frz  = combo['freeze_layers']

    print(f"\n lr={lr}, bs={bs}, warmup={wr}, epochs={ep}, freeze={frz}")
    t_combo = time.time()

    # Bangun model dengan mean-pooling head
    # Deepcopy mencegah fine-tuning satu combo mencemari bobot awal combo berikutnya.
    base_copy = copy.deepcopy(base_model_C)
    model_c   = RoBERTaMeanPoolHead(base_copy, num_labels=3, dropout_rate=0.2)

    # Freeze dinamis: freeze layer 0 .. (frz-1)
    # freeze_layers=4 → train layer 4-11 + head
    # freeze_layers=6 → train layer 6-11 + head
    # Freeze layer bawah mengurangi VRAM/risiko overfit sambil tetap melatih layer atas yang lebih task-specific.
    for name, param in model_c.named_parameters():
        if 'classifier' in name:
            param.requires_grad = True                   # head selalu trainable
            continue
        if 'pooler' in name:
            param.requires_grad = True
            continue
        if 'embeddings' in name:
            param.requires_grad = False                  # embeddings selalu freeze
            continue
        if 'encoder.layer.' in name:
            layer_num = int(name.split('encoder.layer.')[1].split('.')[0])
            if layer_num < frz:
                param.requires_grad = False
            else:
                param.requires_grad = True

    trainable = sum(p.numel() for p in model_c.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model_c.parameters())
    print(f"    Trainable: {trainable/1e6:.1f}M / {total/1e6:.1f}M "
          f"({100*trainable/total:.1f}%)")

    # Training Arguments
    # Gradient accumulation menjaga effective batch besar meski batch per device dibatasi VRAM.
    grad_accum = 2 if bs == 16 else 1                    # effective bs ≈ 32
    steps_per_epoch = math.ceil(len(train_dataset_C) / (bs * grad_accum))
    total_steps     = steps_per_epoch * ep
    warmup_steps    = max(1, int(total_steps * wr))

    # TrainingArguments mengatur strategi evaluasi, checkpoint, scheduler, dan optimasi mixed precision.
    args_c = TrainingArguments(
        output_dir=f'./results_C_lr{lr}_bs{bs}_frz{frz}',
        num_train_epochs=ep,
        per_device_train_batch_size=bs,
        per_device_eval_batch_size=128,
        # fp16 memanfaatkan Tensor Cores T4 untuk mempercepat training dan menghemat memori.
        fp16=True,
        learning_rate=lr,
        weight_decay=0.03,
        # Warmup mencegah update besar di awal fine-tuning saat head masih belum stabil.
        warmup_steps=warmup_steps,
        max_grad_norm=0.5,
        gradient_accumulation_steps=grad_accum,
        lr_scheduler_type="cosine",
        eval_strategy="epoch",
        save_strategy="epoch",                           # aktifkan utk early stop
        logging_steps=50,
        report_to="none",
        dataloader_num_workers=2,
        dataloader_pin_memory=True,
        dataloader_persistent_workers=True,
        load_best_model_at_end=True,                     # ambil best checkpoint
        metric_for_best_model="eval_f1_macro",
        greater_is_better=True,
        disable_tqdm=False,
        seed=42,
    )

    # Early stopping: berhenti jika 2 epoch berturut tidak membaik
    early_stop_cb = EarlyStoppingCallback(early_stopping_patience=2)

    # Trainer menerima class_weights dan smoothing agar loss custom aktif selama train/eval.
    trainer_c = WeightedLabelSmoothingTrainer(
        model=model_c,
        args=args_c,
        train_dataset=train_dataset_C,
        eval_dataset=val_dataset_C,
        compute_metrics=compute_metrics,
        class_weights=class_weights_C,
        smoothing=0.05,
        base_lr=lr,                                      # untuk LLRD
        callbacks=[early_stop_cb],
    )
    trainer_c.train()

    val_metrics = trainer_c.evaluate(eval_dataset=val_dataset_C)
    elapsed = time.time() - t_combo
    print(f"    Val F1: {val_metrics['eval_f1_macro']:.4f} | "
          f"Val Acc: {val_metrics['eval_accuracy']:.4f} | "
          f"Time: {elapsed:.1f}s")

    # Model terbaik dipilih berdasarkan validation macro F1 karena targetnya seimbang antar kelas.
    if val_metrics['eval_f1_macro'] > best_val_f1_C:
        best_val_f1_C  = val_metrics['eval_f1_macro']
        best_trainer_C = trainer_c
        best_params_C  = combo

    # Bersihkan VRAM antar combo
    # Objek besar dihapus antar combo supaya checkpoint berikutnya tidak kehabisan VRAM/RAM.
    del model_c, trainer_c, base_copy
    torch.cuda.empty_cache()
    gc.collect()

print_best("Model C: RoBERTa", best_params_C, "Val F1", best_val_f1_C,
           time.time()-t_start_C)

# Evaluasi akhir
# Evaluasi akhir memakai trainer terbaik yang sudah memuat checkpoint terbaik dari validation.
test_pred_C    = best_trainer_C.predict(test_dataset_C)
y_pred_test_C  = np.argmax(test_pred_C.predictions, axis=-1)
y_pred_train_C = np.argmax(
    best_trainer_C.predict(train_dataset_C).predictions, axis=-1
)

res_C = log_result(
    "Model C: RoBERTa",
    f"{int((1-SPLIT_TEST_RATIO_C)*100)}:{int(SPLIT_TEST_RATIO_C*100)}",
    "RoBERTa+MeanPool+LLRD+LabelSmooth+EarlyStop",
    y_tr_C.values, y_pred_train_C,
    y_te_C.values, y_pred_test_C,
    best_params_C,
)
print(f"  Test Acc: {res_C['Test Accuracy']:.4f} | F1: {res_C['F1-Score']:.4f}")
plot_confusion(y_te_C.values, y_pred_test_C,
               "Confusion Matrix — Model C (RoBERTa)")

In [9]:
# CELL 9: TABEL PERBANDINGAN + VISUALISASI + INFERENCE HASIL + ENSEMBLE VOTING

torch.cuda.empty_cache()

# Tabel awal menampilkan performa model individual sebelum hasil ensemble ditambahkan.
df_comparison = pd.DataFrame(experiment_results).sort_values("Test Accuracy", ascending=False)
print("\nPERBANDINGAN MODEL")
print(df_comparison.to_string(index=False))

# Ensemble digunakan untuk mengurangi kelemahan satu model dengan menggabungkan prediksi tiga arsitektur.
# ENSEMBLE: Majority Voting dari 3 model
# Gunakan test set dari Model C (split 75:25) sebagai acuan
print("\n" + "="*60)
print("ENSEMBLE: Majority Voting (3 Model)")
print("="*60)

# Karena split berbeda, gunakan test set Model C
# Prediksi ulang semua test set dengan semua model
# Untuk fairness, gunakan test set yang sama (dari Model C)
# Test set Model C dipakai sebagai acuan bersama agar voting dihitung pada sampel yang sama.
y_test_ensemble = y_te_C.values

# Prediksi Model A pada test set C
# Model A dan B diprediksi ulang pada X_te_C supaya semua kandidat voting sejajar.
y_pred_A_on_C = predict_dl_model(best_model_A, TextCNNDataset, X_te_C, vocab=vocab_A, max_len=MAX_LEN_A)
y_pred_B_on_C = predict_dl_model(best_model_B, BiLSTMDataset, X_te_C, vocab=vocab_B, max_len=MAX_LEN_B)
y_pred_C_on_C = y_pred_test_C

# Majority Voting
# scipy.stats.mode mengambil label yang paling banyak dipilih per data; dengan 3 model jarang terjadi seri.
from scipy import stats
stacked = np.stack([y_pred_A_on_C, y_pred_B_on_C, y_pred_C_on_C], axis=0)
y_pred_ensemble, _ = stats.mode(stacked, axis=0)
y_pred_ensemble = y_pred_ensemble.flatten()

# Metrik ensemble dihitung dengan macro average agar kontribusi tiap kelas tetap terlihat.
ens_acc = accuracy_score(y_test_ensemble, y_pred_ensemble)
ens_f1  = f1_score(y_test_ensemble, y_pred_ensemble, average='macro', zero_division=0)
ens_prec = precision_score(y_test_ensemble, y_pred_ensemble, average='macro', zero_division=0)
ens_rec  = recall_score(y_test_ensemble, y_pred_ensemble, average='macro', zero_division=0)

print(f"  Ensemble Accuracy : {ens_acc:.4f}")
print(f"  Ensemble Precision: {ens_prec:.4f}")
print(f"  Ensemble Recall   : {ens_rec:.4f}")
print(f"  Ensemble F1-Score : {ens_f1:.4f}")

# Tambahkan ke tabel
# Hasil ensemble memakai format dict yang sama dengan log_result agar bisa digabung ke tabel final.
ensemble_result = {
    "Model": "ENSEMBLE (Voting)", "Split": "75:25",
    "Fitur": "Majority Voting 3 Model",
    "Best Params": "TextCNN + BiLSTM+Attn + RoBERTa",
    "Train Accuracy": "-", "Test Accuracy": ens_acc,
    "Precision": ens_prec, "Recall": ens_rec, "F1-Score": ens_f1,
}
experiment_results.append(ensemble_result)

df_final = pd.DataFrame(experiment_results).sort_values("Test Accuracy", ascending=False)
print("\nHASIL AKHIR (termasuk ENSEMBLE)")
print(df_final.to_string(index=False))

# Plot perbandingan
# Plot bar memudahkan perbandingan cepat antara accuracy dan macro F1 tiap model.
fig, ax = plt.subplots(figsize=(10, 6))
df_final.set_index("Model")[["Test Accuracy", "F1-Score"]].plot(
    kind='bar', ax=ax, color=['#4C72B0', '#DD8452'], rot=20
)
ax.set_title("Perbandingan Model")
ax.set_ylabel("Score"); ax.set_ylim(0.5, 1.0)
plt.tight_layout(); plt.show()

plot_confusion(y_test_ensemble, y_pred_ensemble, "Confusion Matrix — ENSEMBLE")

# WordCloud
# WordCloud membantu inspeksi kualitatif kata dominan pada label hasil weak labeling.
for label_value, cmap in [('positive','Greens'), ('neutral','Greys'), ('negative','Reds')]:
    """Visualisasi WordCloud untuk setiap kelas sentimen.
    
    WordCloud menampilkan kata-kata yang paling sering muncul untuk setiap kelas sentimen:
    - Positive: warna hijau
    - Neutral: warna abu-abu
    - Negative: warna merah
    
    Args:
        label_value (str): Kelas sentimen ('positive', 'neutral', 'negative')
        cmap (str): Colormap untuk visualisasi
    """
    text_subset = ' '.join(df_raw[df_raw['polarity'] == label_value]['text_clean'])
    wc = WordCloud(width=800, height=400, background_color='white', colormap=cmap).generate(text_subset)
    plt.figure(figsize=(8, 4))
    plt.imshow(wc, interpolation='bilinear'); plt.axis('off')
    plt.title(f"WordCloud — '{label_value}'"); plt.show()

# Inference
# Fungsi inference contoh menjalankan pipeline yang sama: clean → ids → logits → probabilitas.
def predict_textcnn(text_review, model, vocab, max_len):
    """Melakukan prediksi sentimen pada teks ulasan menggunakan model TextCNN.
    
    Fungsi ini membersihkan teks, mengonversi ke urutan indeks, dan melakukan prediksi
    menggunakan model TextCNN. Mengembalikan label sentimen dan confidence score.
    
    Args:
        text_review (str): Teks ulasan yang akan diprediksi
        model (nn.Module): Model TextCNN yang sudah dilatih
        vocab (dict): Vocabulary untuk konversi teks ke indeks
        max_len (int): Panjang maksimum urutan
        
    Returns:
        tuple: (label, confidence) - label sentimen dan confidence score
    """
    # Cleaning inference harus sama dengan training agar distribusi input konsisten.
    cleaned = clean_text_light(text_review)
    ids = text_to_ids(cleaned, vocab, max_len)
    x = torch.tensor([ids], dtype=torch.long).to(DEVICE)
    model.eval()
    # no_grad dan autocast mempercepat inference karena tidak ada backward pass.
    with torch.no_grad(), torch.amp.autocast('cuda'):
        logits = model(x)
        probs = torch.softmax(logits.float(), dim=-1).squeeze().cpu().numpy()
    pred_idx = int(np.argmax(probs))
    return ID2LABEL[pred_idx], float(probs[pred_idx])

# Contoh ulasan mencakup positif, negatif, dan netral untuk sanity check manual.
test_reviews = [
    "The storyline is super amazing and the English voice acting is top-tier!",
    "Too many bugs after the update, the game keeps crashing on the loading screen.",
    "The game is okay, average gacha mechanics."
]

print("\n=== HASIL INFERENCE (Model A: TextCNN) ===")
for review in test_reviews:
    sentimen, conf = predict_textcnn(review, best_model_A, vocab_A, MAX_LEN_A)
    print(f'Ulasan   : "{review}"')
    print(f"Sentimen : {sentimen.upper()} (Confidence: {conf*100:.2f}%)")
    print("-" * 50)

# Ringkasan memori akhir membantu mendeteksi sisa alokasi GPU setelah semua evaluasi selesai.
if torch.cuda.is_available():
    print(f"\nGPU Memory: {torch.cuda.memory_allocated()/1e9:.2f} GB / "
          f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")